# Modeling Baselines


In this baseline modeling process, we perform the following sequence of steps:

1. We first preprocess the data. 

2. Impute missing values using the most frequent value for categorical variables and the median for numerical variables.

3. Numerical columns are then rescaled so they are distributed like N(0,1).

4. Perform a stratified split that preserves the grouping by patient_nbr since patients can have repeat visits to the hospital.

5. Fit a logistic regression model through the above pipeline and the use it to make predictions. 

6. Evaluate the performance of the model using various metrics. Recall from our EDA that our target metric is F1-Score.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_recall_curve, 
                             precision_score, recall_score, f1_score, confusion_matrix, 
                             log_loss, average_precision_score, classification_report)

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../data/diabetic_data.csv')

display(df.head())
print(df.columns)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')


In [3]:
mapping_admission_type_id ={
    1: "Emergency",
    2: "Urgent",
    3: "Elective",
    4: "Newborn",
    5: "Not Available",
    6: "NULL",
    7: "Trauma Center",
    8: "Not Mapped"
}


mapping_discharge_disposition_id = {
    1: "Discharged to home",
    2: "Discharged/transferred to another short term hospital",
    3: "Discharged/transferred to SNF",
    4: "Discharged/transferred to ICF",
    5: "Discharged/transferred to another type of inpatient care institution",
    6: "Discharged/transferred to home with home health service",
    7: "Left AMA",
    8: "Discharged/transferred to home under care of Home IV provider",
    9: "Admitted as an inpatient to this hospital",
    10: "Neonate discharged to another hospital for neonatal aftercare",
    11: "Expired",
    12: "Still patient or expected to return for outpatient services",
    13: "Hospice / home",
    14: "Hospice / medical facility",
    15: "Discharged/transferred within this institution to Medicare approved swing bed",
    16: "Discharged/transferred/referred another institution for outpatient services",
    17: "Discharged/transferred/referred to this institution for outpatient services",
    18: "NULL",
    19: "Expired at home. Medicaid only, hospice.",
    20: "Expired in a medical facility. Medicaid only, hospice.",
    21: "Expired, place unknown. Medicaid only, hospice.",
    22: "Discharged/transferred to another rehab fac including rehab units of a hospital",
    23: "Discharged/transferred to a long term care hospital",
    24: "Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare",
    25: "Not Mapped",
    26: "Unknown/Invalid",
    27: "Discharged/transferred to a federal health care facility",
    28: "Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital",
    29: "Discharged/transferred to a Critical Access Hospital (CAH)",
    30: "Discharged/transferred to another Type of Health Care Institution not Defined Elsewhere",
}

mapping_admission_source_id = {
    1: "Physician Referral",
    2: "Clinic Referral",
    3: "HMO Referral",
    4: "Transfer from a hospital",
    5: "Transfer from a Skilled Nursing Facility (SNF)",
    6: "Transfer from another health care facility",
    7: "Emergency Room",
    8: "Court/Law Enforcement",
    9: "Not Available",
    10: "Transfer from critical access hospital",
    11: "Normal Delivery",
    12: "Premature Delivery",
    13: "Sick Baby",
    14: "Extramural Birth",
    15: "Not Available",
    17: "NULL",
    18: "Transfer From Another Home Health Agency",
    19: "Readmission to Same Home Health Agency",
    20: "Not Mapped",
    21: "Unknown/Invalid",
    22: "Transfer from hospital inpt/same fac reslt in a sep claim",
    23: "Born inside this hospital",
    24: "Born outside this hospital",
    25: "Transfer from Ambulatory Surgery Center",
    26: "Transfer from Hospice",
}

In [4]:
# Data preprocessing based off of previous EDA
# Will put this Data Preprocessing into a function to be used in the final model

# Basic preprocessing function to clean data.

def preprocess_data(df):

    # Drop encounter_id
    df = df.drop(['encounter_id'], axis=1)
    df = df.replace('?', pd.NA)

    # Lowercase all Column Names
    df.columns = df.columns.str.lower() 

    # Edit the readmitted column to be a binary target. We are focused on patients readmitted within 30 days.
    df["readmitted"] = df["readmitted"].replace({">30": 0, "<30": 1, "NO": 0})

    df.rename(columns={"readmitted":"target"},inplace=True)
    df["target"].astype(int)
    df["admission_type_id"] = df["admission_type_id"].map(mapping_admission_type_id)
    df["discharge_disposition_id"] = df["discharge_disposition_id"].map(mapping_discharge_disposition_id)
    df["admission_source_id"] =df["admission_source_id"].map(mapping_admission_source_id)
    return df

# Extract numerical and categorical columns

def num_cat(df,target):
    # Identify numerical and categorical columns
    df = df.drop(columns=[target])
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    return num_cols, cat_cols

In [5]:
df = preprocess_data(df)

In [6]:
num_cols, cat_cols = num_cat(df,"target")
print(num_cols)
print(cat_cols)
features = num_cols+cat_cols

X = df[features]
y = df['target'].astype(int) 

['patient_nbr', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'a1cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesmed']


/var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_88090/3754800949.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


In [7]:
# Quick check of columns not used in our features list. This should only return ['target']
# list(set(features) ^ set(df.columns.to_list())) == ['target']

In [8]:
# The target is imbalanced. We also want to keep repeat patients together.

# 1. Set n_splits=5 so the test size naturally becomes 1/5 = 20%
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Get the indices for the very first split next() gets only the first split for us
train_idx, test_idx = next(sgkf.split(X, y, groups=X['patient_nbr']))

# 3. Create a single train/test split
Xt, Xv = X.iloc[train_idx], X.iloc[test_idx]
yt, yv = y.iloc[train_idx], y.iloc[test_idx]

In [9]:
# We will save the train / test split here and use it for the modeling experiments

train = Xt.copy()
train["target"] = yt
train.to_csv("../data/cleaned_data/train.csv",index=False)

test = Xv.copy()
test["target"] = yv 
test.to_csv("../data/cleaned_data/test.csv",index=False)

train.drop(columns=['patient_nbr'])
test.drop(columns=['patient_nbr'])

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,payer_code,medical_specialty,diag_1,diag_2,diag_3,max_glu_serum,a1cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesmed,target
9,12,33,3,18,0,0,0,8,Caucasian,Female,[90-100),NaN,Elective,Discharged/transferred to SNF,Transfer from a hospital,NaN,InternalMedicine,434,198,486,NaN,NaN,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,0
12,7,60,0,15,0,1,0,8,Caucasian,Female,[40-50),NaN,Emergency,Discharged/transferred to SNF,Emergency Room,NaN,Family/GeneralPractice,428,250.43,250.6,NaN,NaN,Steady,Up,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,1
13,10,55,1,31,0,0,0,8,Caucasian,Male,[80-90),NaN,Emergency,Discharged/transferred to home with home healt...,Emergency Room,NaN,Family/GeneralPractice,428,411,427,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,0
15,12,75,5,13,0,0,0,9,AfricanAmerican,Male,[60-70),NaN,Emergency,Discharged/transferred to SNF,Emergency Room,NaN,NaN,999,507,996,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0
21,2,36,2,11,0,0,0,6,NaN,Male,[60-70),NaN,Urgent,Discharged to home,Transfer from a hospital,NaN,NaN,572,456,427,NaN,NaN,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101729,4,35,0,14,0,0,0,9,Caucasian,Female,[70-80),NaN,Elective,Discharged to home,Physician Referral,MC,NaN,276,466,401,NaN,NaN,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,0
101736,8,54,1,20,1,0,1,9,Caucasian,Female,[60-70),NaN,Emergency,Discharged/transferred to SNF,Emergency Room,HM,NaN,590,564,250.8,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,0
101741,4,21,1,8,0,0,1,9,AfricanAmerican,Male,[90-100),NaN,Emergency,Discharged/transferred to SNF,Emergency Room,MC,NaN,820,427,276,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,0
101753,13,51,2,13,0,0,0,9,Other,Male,[40-50),NaN,Emergency,Discharged to home,Emergency Room,NaN,NaN,250.8,730,731,NaN,NaN,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,0


In [10]:
# We will make a Pipeline for the Baseline Model with a Preprocessing Step and a Logistic Regression Model.
# The preprocessing should only be done using the training data.

# A naive baseline will be to simply use a median and most frequent imputer for the numerical and categorical columns respectively.

num = Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num",num,num_cols),("cat",cat,cat_cols)])
clf = Pipeline([("prep", prep),("model", LogisticRegression(max_iter=1000))])


In [11]:
# Fit the model to the training data

clf.fit(Xt,yt)

# Have the model now make predictions

labels = clf.predict(Xv)                   
probs = clf.predict_proba(Xv)
proba = probs[:, 1]

In [12]:
threshold = 0.5
y_pred = (proba >= threshold).astype(int)

print("ROC-AUC:", round(roc_auc_score(yv, proba), 4))
print("PR-AUC: ", round(average_precision_score(yv, proba), 4))
print(f"Accuracy: {accuracy_score(yv, y_pred):.3f}")
print("\nConfusion matrix:\n", confusion_matrix(yv, y_pred))
print(f"\n\n{classification_report(yv, y_pred)}")
print(f"\nF1 @{threshold}: {round(f1_score(yv,y_pred),4)}")

# Best-threshold F1 (optimistic: threshold chosen on Xv itself)
p, r, thr = precision_recall_curve(yv, proba)
f1s = 2 * p * r / (p + r + 1e-12)
best = f1s[:-1].argmax()
print(f"\nBest-threshold F1: {f1s[best]:.4f} at threshold {thr[best]:.4f}")

ROC-AUC: 0.6632
PR-AUC:  0.2112
Accuracy: 0.887

Confusion matrix:
 [[18011    71]
 [ 2234    38]]


              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18082
           1       0.35      0.02      0.03      2272

    accuracy                           0.89     20354
   macro avg       0.62      0.51      0.49     20354
weighted avg       0.83      0.89      0.84     20354


F1 @0.5: 0.0319

Best-threshold F1: 0.2773 at threshold 0.1380


In [13]:
print(f"If we were to simply predict that the discharged patient would not return within")
print(f"30 days no matter what the model would get it correct {100*np.round(1 - sum(yv)/len(yv),3)}% of the time.")

If we were to simply predict that the discharged patient would not return within
30 days no matter what the model would get it correct 88.8% of the time.


# Conclusion

We see here that we have a very low F1-score of 0.245. Our precision is also a bit low. The accuracy is naturally very high and similar to the dummy guess which gets 0.888 due to the unbalanced nature of the target. The AUC score is 0.637 which indicates that the model performs better than a 50/50 guess, but this is not shocking due to the unbalanced nature.

What this shows is that the baseline model is pretty bad and there can be room for significant improvement.